# Fine-tuning a Pairwise SQL Selection Model

**Goal.** Train the highest-leverage missing component of the AEGIS v1 text-to-SQL
pipeline: a **binary pairwise selector** that, given a question and two candidate SQL
queries (plus their execution results), picks the one that answers correctly.
CHASE-SQL's ablation showed a fine-tuned pairwise selector beats plain
self-consistency voting by **+4.17 EX** on BIRD — the single largest documented
selection gain ([CHASE-SQL, arXiv:2410.01943](https://arxiv.org/abs/2410.01943)).

**Pipeline** (each stage checkpoints to disk and is resumable):

```
1. Download BIRD train set (questions + SQLite databases)
2. Generate n candidate SQLs per question with the AEGIS generator
   (cycloneboy/CscSQL-Grpo-Qwen2.5-Coder-7B-Instruct)  ..............  GPU, hours
3. Execute candidates + gold vs the databases -> correct/incorrect labels
4. Build order-randomized (A,B) comparison pairs, split by database
5. LoRA fine-tune Qwen2.5-Coder-3B-Instruct on single-token A/B supervision
6. Evaluate pairwise accuracy on held-out databases
7. Push adapter + merged model to the Hugging Face Hub
8. Inference demo straight from the Hub + AEGIS v1 integration snippet
```

**Expected wall-times** (1x 40-80GB GPU): generation ~3-5 h at 2,000 questions
(set `N_QUESTIONS = 100` for a pilot first), labeling ~15 min, training ~1-2 h.


## 1. Environment

Installs are version-floored (not hard-pinned) so security patches flow, and cached
by pip. Log into Hugging Face with a **write** token — needed for the final push.

In [ ]:
%pip install -q "torch>=2.1" "transformers>=4.44" "peft>=0.11" "trl>=0.9" \
    "datasets>=2.19" "accelerate>=0.30" "huggingface_hub>=0.23" func_timeout tqdm

import torch

assert torch.cuda.is_available(), "A CUDA GPU is required for generation + training."
print(f"GPU: {torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

In [ ]:
# Authenticate to the Hugging Face Hub (write scope, for the deployment step).
# Uses the HF_TOKEN env var when set; otherwise opens the interactive login widget.
import os

from huggingface_hub import login, notebook_login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
    print("Logged in from HF_TOKEN")
else:
    notebook_login()

## 2. Configuration

Every knob lives here; later cells read **only** from `CFG`. For a pilot run set
`N_QUESTIONS = 100` (proves the pipeline end-to-end in ~1 GPU-hour), then rerun the
generation cell with 2,000 — it resumes, so pilot work is never wasted.

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
import random

import numpy as np
import torch


@dataclass(frozen=True)
class Config:
    # --- data -----------------------------------------------------------------
    data_dir: Path = Path("data")
    bird_train_url: str = "https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip"
    n_questions: int = 2000          # BIRD-train sample size (100 = pilot)
    # --- candidate generation ---------------------------------------------------
    generator_model: str = "cycloneboy/CscSQL-Grpo-Qwen2.5-Coder-7B-Instruct"
    n_candidates: int = 8            # samples per question (diversity for pairs)
    gen_temperature: float = 0.8     # CSC-SQL's sampling temperature
    gen_max_tokens: int = 1024
    gen_batch_size: int = 8          # questions per generation batch (HF path)
    exec_timeout_s: int = 30         # per-SQL execution timeout when labeling
    # --- pairs -------------------------------------------------------------------
    max_pairs_per_question: int = 4  # cap so easy questions don't dominate
    val_fraction: float = 0.12       # of DATABASES (split by db => no leakage)
    result_preview_rows: int = 5     # execution-result rows shown to the selector
    # --- fine-tuning ---------------------------------------------------------------
    base_model: str = "Qwen/Qwen2.5-Coder-3B-Instruct"
    lora_r: int = 16                 # CHASE-SQL's setting
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    epochs: int = 3
    lr: float = 1e-4
    train_batch_size: int = 4
    grad_accum: int = 4
    max_seq_len: int = 3072          # schema slice + two candidates fits comfortably
    # --- deployment -------------------------------------------------------------
    hf_user: str = "Daveonyango254"  # <- your HF username
    adapter_repo: str = "aegis-sql-selector-3b-lora"
    merged_repo: str = "aegis-sql-selector-3b"
    seed: int = 42


CFG = Config()
CFG.data_dir.mkdir(exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
print(CFG)

## 3. Download BIRD train

~2 GB zip (questions + `train_databases`). Skipped when already present, so the
cell is rerun-safe. BIRD publishes archives with varying nesting, so after
extraction we *search* for `train.json` and the databases directory instead of
hard-coding paths.

In [ ]:
import zipfile
from urllib.request import urlretrieve

from tqdm.auto import tqdm


def download(url: str, dest: Path) -> Path:
    """Download with a progress bar; skip when the file already exists."""
    if dest.exists():
        print(f"cached: {dest}")
        return dest
    bar = tqdm(unit="B", unit_scale=True, desc=dest.name)

    def hook(blocks, block_size, total):
        bar.total = total
        bar.update(blocks * block_size - bar.n)

    urlretrieve(url, dest, reporthook=hook)
    bar.close()
    return dest


def extract_all(zip_path: Path, out_dir: Path) -> None:
    """Extract recursively: BIRD zips sometimes contain inner zips."""
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    for inner in out_dir.rglob("*.zip"):
        if inner != zip_path:
            with zipfile.ZipFile(inner) as z:
                z.extractall(inner.parent)


train_zip = download(CFG.bird_train_url, CFG.data_dir / "train.zip")
marker = CFG.data_dir / ".extracted"
if not marker.exists():
    extract_all(train_zip, CFG.data_dir)
    marker.touch()

# Locate the pieces regardless of the archive's internal layout.
TRAIN_JSON = next(p for p in CFG.data_dir.rglob("train.json") if "database" not in str(p))
DB_ROOT = next(p for p in CFG.data_dir.rglob("train_databases") if p.is_dir())
print(f"train.json: {TRAIN_JSON}\ndatabases:  {DB_ROOT}")

## 4. Load + sample questions

BIRD-train has ~9,428 questions across ~69 databases (no difficulty labels, unlike
dev) — so we stratify the sample **by database**, which also guarantees every
training database can contribute pairs.

In [ ]:
import json
from collections import defaultdict

rows = json.loads(Path(TRAIN_JSON).read_text())
for i, r in enumerate(rows):          # train.json has no stable id -> assign one
    r["qid"] = i
print(f"BIRD-train: {len(rows)} questions, {len({r['db_id'] for r in rows})} databases")

# Keep only questions whose database file actually exists (a few are broken).
rows = [r for r in rows if (DB_ROOT / r["db_id"] / f"{r['db_id']}.sqlite").exists()]

# Stratified sample: proportional per database, seeded and deterministic.
by_db = defaultdict(list)
for r in rows:
    by_db[r["db_id"]].append(r)
rng = random.Random(CFG.seed)
frac = min(1.0, CFG.n_questions / len(rows))
sample = []
for db, qs in sorted(by_db.items()):
    rng.shuffle(qs)
    sample += qs[: max(1, round(len(qs) * frac))]
sample = sample[: CFG.n_questions]
print(f"sampled {len(sample)} questions from {len({r['db_id'] for r in sample})} databases")

## 5. Schema rendering (OmniSQL DDL)

The generator checkpoint was trained on the **OmniSQL prompt**: full-schema DDL with
per-column comments and example values, evidence prepended to the question, and a
`<think>…</think><answer>SQL</answer>` output wrapper. We reproduce it from the
SQLite files directly (PRAGMA metadata + sampled values), cached per database.

In [ ]:
import re
import sqlite3
from functools import lru_cache

OMNISQL_TEMPLATE = """Task Overview:
You are a data science expert. Below, you are provided with a database schema and a natural language question. Your task is to understand the schema and generate a valid SQL query to answer the question.

Database Engine:
SQLite

Database Schema:
{db_details}
This schema describes the database's structure, including tables, columns, primary keys, foreign keys, and any relevant relationships or constraints.

Question:
{question}

Instructions:
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- The generated query should return all of the information asked in the question without any missing or extra information.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
In your answer, please enclose the generated SQL query in a code block:
```sql
-- Your SQL query
```

Take a deep breath and think step by step to find the correct SQL query.
"""

THINK_SUFFIX = (
    "\nShow your work in <think> </think> tags. And return the final SQLite SQL "
    "query that starts with keyword `SELECT` in <answer> </answer> tags, "
    "for example <answer>SELECT AVG(rating_score) FROM movies</answer>."
)


def quote_ident(name: str) -> str:
    return f"`{name}`" if re.search(r"[^\w]", name) else name


@lru_cache(maxsize=None)
def build_db_details(db_id: str, max_examples: int = 3) -> str:
    """Render one database as OmniSQL-style DDL (cached per database)."""
    path = DB_ROOT / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(f"file:{path}?mode=ro", uri=True)
    conn.text_factory = lambda b: b.decode(errors="replace") if isinstance(b, bytes) else b
    ddl = []
    try:
        tables = [r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]
        for t in tables:
            cols = conn.execute(f"PRAGMA table_info({quote_ident(t)})").fetchall()
            fks = conn.execute(f"PRAGMA foreign_key_list({quote_ident(t)})").fetchall()
            lines = []
            for _, name, ctype, _, _, is_pk in cols:
                comment = [name]
                try:  # up to 3 example values, cheap DISTINCT probe
                    vals = [str(v[0]) for v in conn.execute(
                        f"SELECT DISTINCT {quote_ident(name)} FROM {quote_ident(t)} "
                        f"WHERE {quote_ident(name)} IS NOT NULL LIMIT {max_examples}")]
                    if vals:
                        comment.append(f"example: [{', '.join(vals)}]")
                except sqlite3.Error:
                    pass
                lines.append(f"    {quote_ident(name)} {ctype or 'TEXT'}, -- {' | '.join(comment)}")
            pks = [name for _, name, *_rest, is_pk in
                   [(c[0], c[1], c[2], c[3], c[4], c[5]) for c in cols] if is_pk]
            if pks:
                lines.append(f"    PRIMARY KEY ({', '.join(quote_ident(p) for p in pks)}),")
            for fk in fks:  # (id, seq, ref_table, from_col, to_col, ...)
                lines.append(f"    FOREIGN KEY ({quote_ident(fk[3])}) REFERENCES "
                             f"{quote_ident(fk[2])}({quote_ident(fk[4])}),")
            body = "\n".join(lines).rstrip(",")
            ddl.append(f"CREATE TABLE {quote_ident(t)} (\n{body}\n);")
    finally:
        conn.close()
    return "\n\n".join(ddl)


def generation_prompt(question: str, evidence: str, db_id: str) -> str:
    """OmniSQL prompt with evidence PREPENDED to the question (the trained format)."""
    q = f"{evidence.strip()}\n{question.strip()}" if evidence.strip() else question.strip()
    return OMNISQL_TEMPLATE.format(db_details=build_db_details(db_id), question=q) + THINK_SUFFIX


ANSWER_RE = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.IGNORECASE | re.DOTALL)
FENCE_RE = re.compile(r"```sql\s*(.*?)```", re.IGNORECASE | re.DOTALL)


def extract_sql(text: str) -> str:
    """Pull the final SQL out of a model response (answer tags > sql fence > raw)."""
    for pattern in (ANSWER_RE, FENCE_RE):
        m = pattern.findall(text or "")
        if m:
            return m[-1].strip()
    text = (text or "").strip()
    return text if text.upper().startswith(("SELECT", "WITH")) else ""


print(build_db_details(sample[0]["db_id"])[:400], "...")

## 6. Candidate generation *(GPU — the long stage)*

`n_candidates` samples per question at temperature 0.8 from the AEGIS generator.
Uses **vLLM** when installed (~5-10x faster) and falls back to batched
`transformers` sampling otherwise. Results append to `data/candidates.jsonl`
keyed by question id — **interrupt and rerun freely; finished questions are skipped.**

In [ ]:
CANDS_PATH = CFG.data_dir / "candidates.jsonl"

done = set()
if CANDS_PATH.exists():
    with open(CANDS_PATH) as f:
        done = {json.loads(line)["qid"] for line in f}
todo = [r for r in sample if r["qid"] not in done]
print(f"{len(done)} questions already generated, {len(todo)} to go")

if todo:
    prompts = {r["qid"]: generation_prompt(r["question"], r.get("evidence", ""), r["db_id"])
               for r in todo}
    try:  # ---- fast path: vLLM --------------------------------------------------
        from vllm import LLM, SamplingParams

        llm = LLM(model=CFG.generator_model, dtype="float16", gpu_memory_utilization=0.9)
        params = SamplingParams(n=CFG.n_candidates, temperature=CFG.gen_temperature,
                                max_tokens=CFG.gen_max_tokens)
        with open(CANDS_PATH, "a") as f:
            outs = llm.generate([prompts[r["qid"]] for r in todo], params)
            for r, out in zip(todo, outs):
                sqls = [extract_sql(o.text) for o in out.outputs]
                f.write(json.dumps({"qid": r["qid"], "candidates": [s for s in sqls if s]}) + "\n")
    except ImportError:  # ---- portable path: transformers ------------------------
        from transformers import AutoModelForCausalLM, AutoTokenizer

        tok = AutoTokenizer.from_pretrained(CFG.generator_model)
        model = AutoModelForCausalLM.from_pretrained(
            CFG.generator_model, dtype=torch.float16, device_map="auto")
        model.eval()
        with open(CANDS_PATH, "a") as f:
            for r in tqdm(todo, desc="generating"):
                text = tok.apply_chat_template(
                    [{"role": "user", "content": prompts[r["qid"]]}],
                    tokenize=False, add_generation_prompt=True)
                inputs = tok(text, return_tensors="pt", truncation=True,
                             max_length=8192).to(model.device)
                with torch.no_grad():
                    out = model.generate(
                        **inputs, do_sample=True, temperature=CFG.gen_temperature,
                        num_return_sequences=CFG.n_candidates,
                        max_new_tokens=CFG.gen_max_tokens,
                        pad_token_id=tok.eos_token_id)
                sqls = [extract_sql(tok.decode(seq[inputs["input_ids"].shape[1]:],
                                               skip_special_tokens=True)) for seq in out]
                f.write(json.dumps({"qid": r["qid"], "candidates": [s for s in sqls if s]}) + "\n")
                f.flush()  # checkpoint after every question
        del model
        torch.cuda.empty_cache()
print("candidate generation complete")

## 7. Execution labeling

Run the gold SQL and every candidate against the database; a candidate is
**correct** when its result *set* equals gold's (order-independent — the BIRD EX
definition). Timeouts/errors mark a candidate incorrect. Output:
`data/labeled.jsonl` with per-candidate labels + a result preview for the
selector prompt.

In [ ]:
from func_timeout import FunctionTimedOut, func_timeout


def run_sql(db_id: str, sql: str, timeout: int):
    """Execute one query read-only. Returns (frozenset(rows) | None, preview_rows)."""
    path = DB_ROOT / db_id / f"{db_id}.sqlite"

    def _go():
        conn = sqlite3.connect(f"file:{path}?mode=ro", uri=True)
        conn.text_factory = lambda b: b.decode(errors="replace") if isinstance(b, bytes) else b
        try:
            rows = conn.execute(sql).fetchall()
            return frozenset(rows), rows[: CFG.result_preview_rows]
        finally:
            conn.close()

    try:
        return func_timeout(timeout, _go)
    except (FunctionTimedOut, sqlite3.Error, OverflowError):
        return None, None


LABELED_PATH = CFG.data_dir / "labeled.jsonl"
by_qid = {r["qid"]: r for r in sample}
already = set()
if LABELED_PATH.exists():
    with open(LABELED_PATH) as f:
        already = {json.loads(line)["qid"] for line in f}

with open(CANDS_PATH) as f_in, open(LABELED_PATH, "a") as f_out:
    for line in tqdm(list(f_in), desc="labeling"):
        rec = json.loads(line)
        if rec["qid"] in already or rec["qid"] not in by_qid:
            continue
        q = by_qid[rec["qid"]]
        gold_set, _ = run_sql(q["db_id"], q["SQL"], CFG.exec_timeout_s)
        if gold_set is None:      # un-executable gold -> question unusable
            continue
        cands = []
        for sql in dict.fromkeys(rec["candidates"]):        # dedupe, keep order
            res_set, preview = run_sql(q["db_id"], sql, CFG.exec_timeout_s)
            cands.append({
                "sql": sql,
                "correct": res_set is not None and res_set == gold_set,
                "preview": repr(preview)[:300] if preview is not None else "Execution error",
            })
        f_out.write(json.dumps({"qid": rec["qid"], "db_id": q["db_id"],
                                "question": q["question"],
                                "evidence": q.get("evidence", ""),
                                "candidates": cands}) + "\n")
        f_out.flush()

# Headroom check: the selector only helps on questions with BOTH labels present.
labeled = [json.loads(line) for line in open(LABELED_PATH)]
mixed = sum(1 for r in labeled
            if {c["correct"] for c in r["candidates"]} == {True, False})
print(f"labeled {len(labeled)} questions; {mixed} have mixed correct/incorrect "
      f"candidates (these produce training pairs)")

## 8. Comparison pairs

For each mixed question: (correct, incorrect) pairs with **randomized A/B order**
(so the model can't learn a position bias), capped per question, **split
train/val by database** so validation measures generalization to unseen schemas.
The prompt shows both SQLs *with execution previews* — the same evidence the
selector will see inside AEGIS (candidates are grouped by execution result).

In [ ]:
SELECTOR_SYSTEM = (
    "You are an expert SQL judge. Given a database schema, a question, and two "
    "candidate SQLite queries with their execution results, decide which candidate "
    "answers the question correctly. Reply with exactly one letter: A or B."
)


def selector_prompt(question, evidence, db_id, sql_a, prev_a, sql_b, prev_b):
    """The comparison prompt (schema reduced to tables the candidates touch)."""
    tables = set()
    for s in (sql_a, sql_b):
        tables.update(m.group(1).lower() for m in
                      re.finditer(r"(?:from|join)\s+`?([A-Za-z_]\w*)`?", s, re.I))
    ddl = build_db_details(db_id)
    kept = [blk for blk in ddl.split("\n\n")
            if blk.split("(")[0].replace("CREATE TABLE", "").strip(" `").lower() in tables]
    schema = "\n\n".join(kept) if kept else ddl
    q = f"{evidence.strip()}\n{question.strip()}" if evidence.strip() else question.strip()
    return (f"Database schema:\n{schema}\n\nQuestion: {q}\n\n"
            f"Candidate A:\n{sql_a}\nExecution result A: {prev_a}\n\n"
            f"Candidate B:\n{sql_b}\nExecution result B: {prev_b}\n\n"
            f"Which candidate answers the question correctly? Reply with exactly A or B.")


pair_rng = random.Random(CFG.seed)
examples = []
for r in labeled:
    correct = [c for c in r["candidates"] if c["correct"]]
    wrong = [c for c in r["candidates"] if not c["correct"]]
    pairs = [(c, w) for c in correct for w in wrong][: CFG.max_pairs_per_question]
    for c, w in pairs:
        if pair_rng.random() < 0.5:                    # randomize which side is correct
            a, b, label = c, w, "A"
        else:
            a, b, label = w, c, "B"
        examples.append({
            "db_id": r["db_id"],
            "prompt": selector_prompt(r["question"], r["evidence"], r["db_id"],
                                      a["sql"], a["preview"], b["sql"], b["preview"]),
            "label": label,
        })

# Split by DATABASE: validation schemas are never seen in training.
dbs = sorted({e["db_id"] for e in examples})
pair_rng.shuffle(dbs)
val_dbs = set(dbs[: max(1, int(len(dbs) * CFG.val_fraction))])
train_ex = [e for e in examples if e["db_id"] not in val_dbs]
val_ex = [e for e in examples if e["db_id"] in val_dbs]
print(f"{len(train_ex)} train / {len(val_ex)} val pairs "
      f"({len(val_dbs)} held-out databases)")

from datasets import Dataset

train_ds = Dataset.from_list(train_ex)
val_ds = Dataset.from_list(val_ex)
train_ds.save_to_disk(str(CFG.data_dir / "pairs_train"))
val_ds.save_to_disk(str(CFG.data_dir / "pairs_val"))

## 9. LoRA fine-tuning

Single-token supervision (the assistant turn is just `A` or `B`) converges fast and
makes inference a 1-token decode. LoRA r=16 on attention+MLP projections
(CHASE-SQL's setting), bf16, cosine schedule, completion-only loss.

In [ ]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

tok = AutoTokenizer.from_pretrained(CFG.base_model)


def to_chat(example):
    """Render one pair as a chat transcript; the completion is the single label token."""
    return {"text": tok.apply_chat_template(
        [{"role": "system", "content": SELECTOR_SYSTEM},
         {"role": "user", "content": example["prompt"]},
         {"role": "assistant", "content": example["label"]}],
        tokenize=False)}


train_chat = train_ds.map(to_chat, remove_columns=train_ds.column_names)
val_chat = val_ds.map(to_chat, remove_columns=val_ds.column_names)

model = AutoModelForCausalLM.from_pretrained(
    CFG.base_model, dtype=torch.bfloat16, device_map="auto")
model.config.use_cache = False        # incompatible with gradient checkpointing

trainer = SFTTrainer(
    model=model,
    processing_class=tok,
    train_dataset=train_chat,
    eval_dataset=val_chat,
    peft_config=LoraConfig(
        r=CFG.lora_r, lora_alpha=CFG.lora_alpha, lora_dropout=CFG.lora_dropout,
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]),
    args=SFTConfig(
        output_dir="checkpoints",
        num_train_epochs=CFG.epochs,
        learning_rate=CFG.lr,
        per_device_train_batch_size=CFG.train_batch_size,
        gradient_accumulation_steps=CFG.grad_accum,
        gradient_checkpointing=True,
        bf16=True,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        max_length=CFG.max_seq_len,
        dataset_text_field="text",
        report_to="none",
        seed=CFG.seed,
    ),
)
trainer.train()
trainer.save_model("checkpoints/final_adapter")
print("adapter saved -> checkpoints/final_adapter")

## 10. Evaluate pairwise accuracy

Deterministic 1-token decode on the held-out databases. Baselines: random = 50%,
and 'always A' ≈ 50% (order was randomized). CHASE-class selectors land around
**65-75%** pairwise accuracy — enough for the tournament to beat majority voting.

In [ ]:
from tqdm.auto import tqdm


def pick(model, tok, prompt: str) -> str:
    """One-token A/B decision (greedy)."""
    text = tok.apply_chat_template(
        [{"role": "system", "content": SELECTOR_SYSTEM},
         {"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", truncation=True,
                 max_length=CFG.max_seq_len).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=1, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, -1:], skip_special_tokens=True).strip().upper()[:1]


model.eval()
correct = sum(pick(model, tok, e["prompt"]) == e["label"]
              for e in tqdm(val_ex, desc="val"))
acc = correct / max(1, len(val_ex))
print(f"pairwise accuracy on {len(val_ex)} held-out pairs: {acc:.1%}  (random = 50%)")

## 11. Deploy to the Hugging Face Hub

Two artifacts: the **LoRA adapter** (small, cheap to iterate) and the **merged
fp16 model** (drop-in inference with plain `transformers`). The model card records
data provenance, the exact prompt format, and the metric you just measured.

In [ ]:
from huggingface_hub import ModelCard

adapter_id = f"{CFG.hf_user}/{CFG.adapter_repo}"
merged_id = f"{CFG.hf_user}/{CFG.merged_repo}"

CARD = f"""---
license: apache-2.0
base_model: {CFG.base_model}
tags: [text-to-sql, pairwise-selection, bird, lora]
---
# AEGIS SQL Pairwise Selector (3B)

Binary pairwise selection model for text-to-SQL candidate ranking
(CHASE-SQL-style). Given a question, schema, and two candidate SQLite queries
with execution results, it replies `A` or `B` for the correct one.

- Base: `{CFG.base_model}`, LoRA r={CFG.lora_r} merged
- Training pairs: {len(train_ex)} (BIRD-train candidates from
  `{CFG.generator_model}`, execution-labeled, order-randomized, split by database)
- Held-out pairwise accuracy: {acc:.1%} on {len(val_ex)} pairs / {len(val_dbs)} unseen databases

## Prompt format
System: ```{SELECTOR_SYSTEM}```
User: schema + question + `Candidate A/B` + execution previews +
"Which candidate answers the question correctly? Reply with exactly A or B."
Decode 1 token greedily.
"""

# 1) adapter (PEFT weights only)
trainer.model.push_to_hub(adapter_id, private=True)
tok.push_to_hub(adapter_id, private=True)
ModelCard(CARD).push_to_hub(adapter_id)
print(f"adapter  -> https://huggingface.co/{adapter_id}")

# 2) merged fp16 (plain-transformers inference, no peft needed)
merged = trainer.model.merge_and_unload()
merged.push_to_hub(merged_id, private=True)
tok.push_to_hub(merged_id, private=True)
ModelCard(CARD).push_to_hub(merged_id)
print(f"merged   -> https://huggingface.co/{merged_id}")

del merged, model, trainer
torch.cuda.empty_cache()

## 12. Inference from the Hub

Fresh load of the merged model — exactly what AEGIS v1 (or any client) will do.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

sel_tok = AutoTokenizer.from_pretrained(merged_id)
selector = AutoModelForCausalLM.from_pretrained(
    merged_id, dtype=torch.bfloat16, device_map="auto")
selector.eval()

demo = val_ex[0]
choice = pick(selector, sel_tok, demo["prompt"])
print(f"selector picked: {choice}   (gold: {demo['label']})")

### Wiring into AEGIS v1

In `aegis-sql` (branch `aegis_v1`), point the tie-break judge at this model —
the natural upgrade path is replacing `_judge_model` in
`agents/orchestrator.py` with a pairwise **tournament** over the execution-vote
groups (each group's representative + result preview), using this selector:

```yaml
# config.yaml
selection:
  judge: local
# models: add  selector: <your-hf-user>/aegis-sql-selector-3b
```

The tournament runs one 1-token decode per comparison (≤ C(k,2) for k vote
groups, typically ≤ 6 comparisons/query) — negligible latency next to generation,
and per CHASE-SQL worth ~+4 EX over voting alone.